At present PetThermoTools isn't included by default in VICTOR. Luckily, the underlying alphaMELTS for Python package (which requires further, more complex, installation is). To get PetThermoTools working the first thing you need to do is install it via pip!

In [3]:
%pip install petthermotools

**Welcome to this PetThermoTools introduction!** In this notebook we'll walk through a brief fractional crystallization example, and use this to demonstrate how easy it is to get information (e.g., mass, volume, entropy etc.) from the output variables. We'll also explore some in-built plotting functions as well as integration with the Thermobar code of Penny Wieser (Wieser et al. 2022). 

Let's go ahead an import the required packages. When importing PetThermoTools the code will automatically search for the alphaMELTS for Python files. As these are included by default in VICTOR it should print the message: `alphaMELTS for Python files successfully located`, That tells us we're good to go!

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

import petthermotools as ptt
import Thermobar as pt 

ptt.__version__

alphaMELTS for Python files successfully located.
If using the Green et al. (2025) or Weller et al. (2024) thermodynamic models please run `ptt.activate_petthermotools_env()` prior to any calculations.


'0.4.0'

One oddity of PetThermoTools is that when running on Mac or Linux systems (such as VICTOR) every output from the underlying alphaMELTS C code is printed in the notebook. This can be useful for debugging etc. but also means you'll have to scroll through 100s of lines of text outputs to get to the next code cell! 

To avoid this, you can run the two lines below to suppress all outputs. **You have to run this cell twice** for it to work, I'm currently working on a better way to suppress the outputs from alphaMELTS and will post on our Discord page once I've figures it out!

In [3]:
import sys
import os
sys.stdout = open(os.devnull, 'w')
sys.stderr = open(os.devnull, 'w')

For this example we'll use some melt inclusion and submarine matrix glass data from Isla Fernandina in the Galapagos Archipelago. Here we can load in those glass compositions and isolate the most MgO-rich composition to use as the starting point for our crystallization calculations.

In [4]:
df = pd.read_excel("Fernandina_glass.xlsx")
df = df.fillna(0.0)

comp = df.sort_values('MgO', ascending = False, ignore_index = True).loc[0]

comp

Group           MI
Latitude       0.0
Longitude      0.0
SiO2          47.5
TiO2          2.29
Al2O3         16.4
FeOt          9.16
MnO          0.123
MgO           9.38
CaO           11.6
Na2O          2.25
K2O          0.329
P2O5         0.257
H2O           0.68
CO2          0.028
Name: 0, dtype: object

Now that we have a starting composition we can run a number of different thermodynamic calculations in PetThermoTools. Here we'll use that high MgO basaltic composition as the starting point for a crystallization calculation. Run the cells below to perform a single fractional crystallization calculation and examine the results.

In [5]:
Xstal_single = ptt.isobaric_crystallisation(
    Model = "MELTSv1.2.0",
    bulk = comp,
    find_liquidus = True,
    T_end_C = 1050,
    dt_C = 2.0,
    P_bar = 2000,
    Fe3Fet_init = 0.15,
    Frac_solid = True)

In [6]:
Xstal_single.keys()

This demonstrates the typical output structure of PetThermoTools, a Python Dictionary containing DataFrames that contain information about the chemical composition and/or thermodynamic properties of each phase. Let's inspect a few of these!

In [7]:
Xstal_single['liquid1'].head()

We can also use the in-built `harker` function to inspect the composition of the magma during cooling and crystallization.

In [8]:
ptt.harker(Results=Xstal_single)

While useful, a single crystallization calculation might not be the most useful to your study! Quite often users may be interested in investigating how changing the input parameters might influence the liquid line of descent. PetThermoTools makes it easy to do this, here we run a near identical calculation but define `P_bar` as a list of values, rather than a single value. PetThermoTools will take this calculation and run 4 thermodynamic models in parallel. As these calculations are performed simultaneously (up to the number of processors available) this will be significantly quicker than running 4 calculations individually.

In [9]:
Xstal_multi = ptt.isobaric_crystallisation(
    Model = "MELTSv1.2.0",
    bulk = comp,
    find_liquidus = True,
    T_end_C = 1050,
    dt_C = 2.0,
    P_bar = [500,1000,2000,4000],
    Fe3Fet_init = 0.15,
    Frac_solid = True
)

Xstal_multi.keys()

Similar to above, we can plot the results using the `harker` function and also add in the melt inclusion and matrix glass data from Isla Fernandina to compare to our model predictions.

In [10]:
ptt.harker(Results=)

One thing we targeted when developing PetThermoTools was to allow easy integration with other Python packages used for petrological research. One of those is Thermobar, which we imported at the start of the notebook. Let's quickly look at the outputs from one of the calculations.

In [11]:
Xstal_multi['P = 500.0 bars']['clinopyroxene1']

The suffix used in the output DataFrames (e.g., '_Liq', '_Cpx') are the same as those used in Thermobar. This means that the PetThermoTools outputs can easily integrate with the functions of Thermobar to create plots, calculate melt viscosity, etc. As an example, let's plot the composition of clinopyroxene predicted from the crystallization models on a pyroxene quadrilateral:

In [12]:
f, tax = pt.plot_px_classification(figsize = (10,5), major_grid=True, labels = True)
for model in Xstal_multi.keys():
    cpx_comps = pt.tern_points_px(px_comps = )
    tax.scatter(cpx_comps, edgecolor = 'k', s = 40, label = model)

tax.legend()